# Data preparation

In [52]:
import pandas as pd
import numpy as np

Data = pd.read_csv("../data_cleaning/panel_weekly.csv", parse_dates=["week"])
Data = Data.sort_values(["week", "commodity"]).reset_index(drop=True)
 
check = False
if check:
    print(Data.head(10))
    print("\nTypes:\n", Data.dtypes)
    print("\nMissing values per column:\n", Data.isna().sum())
    print("\nDuplicated (week, commodity):", Data.duplicated(["week", "commodity"]).sum())
    print("\nWeeks per commodity:\n", Data.groupby("commodity").size())
    print("\nAll dates are Fridays:", (Data["week"].dt.dayofweek == 4).all())
    print("Period:", Data["week"].min().date(), "->", Data["week"].max().date())

# %% [2] Targets and feature lists
commodities = ["crude_oil", "gold", "copper", "natural_gas", "wheat"]
 
PRICE_FEATURES = ["ret", "mom_4w", "mom_12w", "vol_12w"]
TEXT_FEATURES = ["net_tone", "attention", "has_news"]
CAT_FEATURES = ["commodity"]
# n_relevant is not used: attention = log(1 + n_relevant) carries the same information.
 
FEATURE_SETS = {
    "A_price": PRICE_FEATURES + CAT_FEATURES,
    "B_text":  TEXT_FEATURES + CAT_FEATURES,
    "C_both":  PRICE_FEATURES + TEXT_FEATURES + CAT_FEATURES,
}
 
Data = Data[Data["vol_12w"] > 0].copy()
Data["target_vs"] = Data["target"] / Data["vol_12w"]   # volatility-scaled return (regression target)
Data["up"] = (Data["target"] > 0).astype(int)          # direction (classification target)
Data.head()

#[3] One table per commodity (only if needed, built from the main table)
data_by_commodity = {c: Data[Data["commodity"] == c].reset_index(drop=True) for c in commodities}
Data_crude_oil = data_by_commodity["crude_oil"]        # e.g. for plots or per-commodity models


In [53]:
# %% [4] Train / test split by TIME (never random)
TEST_SHARE = 0.2
weeks = np.sort(Data["week"].unique())
cutoff = weeks[int(len(weeks) * (1 - TEST_SHARE))]
 
train = Data[Data["week"] < cutoff].reset_index(drop=True)   # the past
test = Data[Data["week"] >= cutoff].reset_index(drop=True)   # the future, untouched until the end
 
print(f"Train: {train['week'].min().date()} -> {train['week'].max().date()}  ({len(train)} rows)")
print(f"Test:  {test['week'].min().date()} -> {test['week'].max().date()}  ({len(test)} rows)")
# The target of the last train week (return to the first test Friday) is known at that Friday's
# close, when the first test prediction is made: no overlap between train and test.

# %% [5] Cross-validation folds by week, inside the training set only



from sklearn.model_selection import TimeSeriesSplit
 
 
def time_folds(df, n_splits=5):
    """Expanding-window folds: always train on earlier weeks, validate on later weeks.
    Splits on whole weeks, so the 5 commodities of a week stay together."""
    w = np.sort(df["week"].unique())
    folds = []
    for tr_w, va_w in TimeSeriesSplit(n_splits=n_splits).split(w):
        tr_idx = np.where(df["week"].isin(w[tr_w]))[0]
        va_idx = np.where(df["week"].isin(w[va_w]))[0]
        folds.append((tr_idx, va_idx))
    return folds
 
 
cv = time_folds(train, n_splits=5)
for k, (tr, va) in enumerate(cv):
    print(f"fold {k}: train up to {train['week'].iloc[tr].max().date()}, "
          f"validate {train['week'].iloc[va].min().date()} -> {train['week'].iloc[va].max().date()}")
 

Train: 2010-01-01 -> 2023-05-12  (3490 rows)
Test:  2023-05-19 -> 2026-09-18  (875 rows)
fold 0: train up to 2012-03-30, validate 2012-04-06 -> 2014-06-20
fold 1: train up to 2014-06-20, validate 2014-06-27 -> 2016-09-09
fold 2: train up to 2016-09-09, validate 2016-09-16 -> 2018-11-30
fold 3: train up to 2018-11-30, validate 2018-12-07 -> 2021-02-19
fold 4: train up to 2021-02-19, validate 2021-02-26 -> 2023-05-12


In [54]:
#[6] Preprocessing pipeline (fitted inside each fold, never on all data)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
 
 
def make_preprocess(features, scale=False):
    """Median imputation (+ optional scaling) for numeric columns, one-hot for the commodity.
    Trees (Random Forest, LightGBM) do not need scaling: keep scale=False for them."""
    num = [f for f in features if f not in CAT_FEATURES]
    cat = [f for f in features if f in CAT_FEATURES]
    num_steps = [("impute", SimpleImputer(strategy="median"))] # Step in the pipeline
    if scale:
        num_steps.append(("scale", StandardScaler())) # add this step (scaling). Usefuf for the regression methods (ridge)
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat), #models only accept numbers, and commodity is text (why the OneHotEncoder)
    ]) #The dispatcher: it sends each group of columns to its own treatment, then glues the results side by side.

# handle_unknown="ignore": if a commodity appears that wasn’t seen during training, it gets all zeros instead of crashing.
# sparse_output=False: returns a normal table of numbers rather than a compressed format, which is simpler and works with every model.
 
# Usage later:  model = Pipeline([("prep", make_preprocess(FEATURE_SETS["C_both"])),
#                                 ("reg", RandomForestRegressor(...))])




# 7 Dumb baselines (scores every model must beat), on the validation folds
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score
 
 
def r2_vs_zero(y_true, y_pred):
    """Out-of-sample R2 against a forecast of 0 (standard for return prediction)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return 1 - ((y_true - y_pred) ** 2).sum() / (y_true ** 2).sum()
 
 
r2_zero = make_scorer(r2_vs_zero)
 
X_train = train[FEATURE_SETS["C_both"]]
y_train_reg = train["target_vs"]
y_train_up = train["up"]
 
print("Regression baselines (R2 vs zero, higher is better):")
for name, dummy in {"predict 0": DummyRegressor(strategy="constant", constant=0.0),
                    "predict train mean": DummyRegressor(strategy="mean")}.items():
    s = cross_val_score(dummy, X_train, y_train_reg, cv=cv, scoring=r2_zero)
    print(f"  {name:20s} {s.mean():.4f}")
 
acc = cross_val_score(DummyClassifier(strategy="most_frequent"), X_train, y_train_up,
                      cv=cv, scoring="accuracy")
print(f"\nDirection baseline (always the majority class): accuracy {acc.mean():.3f}")
print(f"Share of 'up' weeks in train: {y_train_up.mean():.3f}   (ROC AUC of any constant guess = 0.5)")

Regression baselines (R2 vs zero, higher is better):
  predict 0            0.0000
  predict train mean   -0.0017

Direction baseline (always the majority class): accuracy 0.481
Share of 'up' weeks in train: 0.510   (ROC AUC of any constant guess = 0.5)


## Summary of the Data Preparation step

**Data.** The prepared panel `panel_weekly.csv` is loaded without modification. It contains one row
per week × commodity (crude oil, natural gas, gold, copper, wheat), from [first week] to [last week],
with [N] weeks per commodity. Checks: no missing values, no duplicated (week, commodity) pairs, all
dates are Fridays, and every commodity covers the same weeks.

**Targets.**
- Regression: next week's return scaled by the 12-week volatility, `target_vs` $= r_{i,t+1} / \sigma_{i,t}$.
  Scaling prevents the most volatile commodity (natural gas) from dominating the fit.
- Direction (for interpretation): `up` $= 1$ if $r_{i,t+1} > 0$, else 0.

**Feature sets.** The same models are trained on three sets, to test whether the text adds
information beyond prices:

| Set | Features |
|---|---|
| A (prices) | `ret`, `mom_4w`, `mom_12w`, `vol_12w`, `commodity` |
| B (text) | `net_tone`, `attention`, `has_news`, `commodity` |
| C (both) | all of the above |

`n_relevant` is not used, since `attention` $= \log(1 + $ `n_relevant`$)$ carries the same information.

**Train / test split by time.** The last 20% of weeks form the test set ([test start] to [test end]),
which is used only once, for the final evaluation. The split is chronological, never random: the
model is always trained on the past and evaluated on the future. The test period lies mostly after
the training cutoff of the LLM used for scoring (about September 2023), which limits its look-ahead bias.

**Cross-validation.** Within the training period, models are compared on 5 expanding-window folds:
each fold trains on all earlier weeks and validates on the following block. Folds are cut on whole
weeks, so the five commodities of the same week always stay together.

**Preprocessing.** Learned transformations are placed in a scikit-learn `Pipeline`, so they are
fitted inside each fold on the training rows only:
- numeric features: median imputation (no effect here, as there are no missing values; kept as a safeguard);
- `commodity`: one-hot encoding into five 0/1 columns, so a single pooled model can learn
  commodity-specific behaviour;
- no scaling, since Random Forest and LightGBM split on thresholds and are insensitive to it.

**Baselines.** Every model must beat three naive forecasts, evaluated on the same folds:
predicting a return of 0, predicting the training mean, and always predicting the majority
direction. Performance is measured by the out-of-sample $R^2$ relative to a zero forecast,
$R^2_{oos} = 1 - \sum (y - \hat y)^2 / \sum y^2$, so a positive value means the model improves on
predicting no movement.

# Applyiing the models